# **Семинар 3.** Рекуррентные нейронные сети. Моделирование языка

## **3.1. Моделирование языка**

Прежде чем говорить о моделировании языка, давайте попробуем разобраться, что этот термин вообще означает. Рассмотрим некоторый текст:

<div style="border-left: 4px solid slategray; padding: 12px 12px 12px 24px; margin: 12px 0; color: slategray; display: flex; align-items: center;">
    <span>
        Однажды такой они подняли крик,<br>
        Что в гневе проснулся косматый старик,<br>
        Великий Колдун, раздражительный дед,<br>
        Проспавший в пещере три тысячи ...<br><br>
        <em>Отрывок из стихотворения Эдуарда Асадова</em>
    </span>
</div>

Попробуйте предположить, какое слово идет в нем следующим. Получилось? Отлично!

Вам был дан некоторый **контекст** — начало предложения. Используя накопленные знания и опыт, вы интуитивно оценили, какое слово **наиболее вероятно** будет следующим.

Так и хорошо обученная языковая модель должна уметь оценивать **распределение вероятностей следующего токена при условии данного контекста**. Тогда для генерации текста можно будет последовательно выбирать токены на основе этих вероятностей и добавлять их к уже полученному тексту.

С примером такого предсказания вы сталкиваетесь постоянно — это подсказки слов в клавиатуре смартфона. Модель смотрит на текст, который вы уже ввели, и предлагает наиболее вероятные продолжения.

А как такую модель обучить? Пусть у нас есть некоторый текст и следующий за ним токен. Для начала давайте поймем, какую задачу мы будем решать. Нам нужно предсказать следующий токен из словаря $V$ размера $|V|$, то есть научиться выбирать один правильный класс из $|V|$ классов.

<div style="border-left: 4px solid dodgerblue; padding: 12px 12px 12px 24px; margin: 12px 0; color: dodgerblue; display: flex; align-items: center;">
    <strong>Предсказание следующего токена можно сформулировать как задачу многоклассовой классификации.</strong>
</div>

Возникает другая проблема: мы помним, что классификация относится к задачам обучения с учителем (англ. *supervised learning*). Для обучения качественной модели нам потребуется огромное количество размеченных данных. Но где их взять?

К счастью, мы можем прибегнуть к следующему трюку: возьмем обычный текст и попробуем по первому токену предсказать второй, по первым двум — третий и так далее. Оказывается, наши сырые данные содержат как признаки, так и правильные ответы!

Такая задача называется **next-token prediction**, а обучение, в котором правильные ответы получаются из самих данных, — **self-supervised learning**. Теперь мы можем просто взять достаточно большой объем текстов и обучить модель на нем.

Обучение больших языковых моделей может занимать месяцы на огромных вычислительных кластерах, а объем собираемых для обучения данных может измеряться десятками и сотнями терабайт (1 TB = 1000 GB).

Чтобы завершить обучение в разумные сроки, но при этом получить хороший результат, нам нужна небольшая модель и тексты с относительно простой структурой. Поэтому сегодня мы будем использовать **символьную RNN** (англ. *character-level RNN*) для генерации названий микроорганизмов.

In [ ]:
!head data/microorganism_names.txt

Для начала соберем словарь. Он будет включать буквы английского алфавита, пробел, дефис и несколько специальных токенов:

- `[BOS]` — *beginning of sequence*, токен начала последовательности, с него начинается генерация;
- `[EOS]` — *end of sequence*, токен окончания последовательности, он означает завершение генерации;
- `[PAD]` — *padding*, его предназначение обсудим чуть позже.

In [ ]:
from string import ascii_letters

SPECIAL_TOKENS = ["[BOS]", "[EOS]", "[PAD]"]

VOCABULARY = SPECIAL_TOKENS + list(ascii_letters + "- ")
VOCABULARY[:10]

Создадим два вспомогательных словаря и две функции.

In [ ]:
chr2idx = {chr: idx for idx, chr in enumerate(VOCABULARY)}


def encode(text: str) -> list[int]:
    return [chr2idx[chr] for chr in ("[BOS]", *list(text), "[EOS]")]


encode("Escherichia coli")

In [ ]:
idx2chr = {idx: chr for chr, idx in chr2idx.items()}


def decode(encoded: list[int]) -> str:
    return "".join([idx2chr[idx] for idx in encoded])


decode([0, 33, 21, 5, 10, 7, 20, 11, 5, 10, 11, 3, 56, 5, 17, 14, 11, 1])

Напишем класс датасета для работы с данными.

In [ ]:
import os

import torch
from torch.utils.data import Dataset


class MicroDataset(Dataset):
    def __init__(self, path: str) -> None:
        super().__init__()

        if not os.path.exists(path):
            msg = f"failed to load dataset from {path}. File does not exist."
            raise FileNotFoundError(msg)

        with open(path) as file:
            self._data = list(map(encode, file.read().splitlines()))

    def __getitem__(self, idx: int) -> torch.Tensor:
        return torch.tensor(self._data[idx])

    def __len__(self) -> int:
        return len(self._data)


ds = MicroDataset("data/microorganism_names.txt")
ds[42]

In [ ]:
decode(ds[42].tolist())

Обычно мы оборачиваем датасет в `DataLoader`, который формирует батчи для обучения модели. Однако есть проблема:

In [ ]:
[len(ds[i]) for i in range(10)]

Разные тексты имеют разную длину, поэтому мы не можем сразу объединить их в один тензор формы $(B, L)$. Здесь нам и пригодится токен `[PAD]`: дополним им более короткие последовательности до длины самой длинной последовательности в батче.

In [ ]:
from torch.nn.utils.rnn import pad_sequence


def collate(batch: list[torch.Tensor]) -> torch.Tensor:
    return pad_sequence(batch, batch_first=True, padding_value=chr2idx["[PAD]"])


collate([ds[i] for i in range(4)])

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(ds, batch_size=32, shuffle=True, collate_fn=collate)

next(iter(dataloader))

## **3.2. Рекуррентная нейронная сеть**

Теперь мы готовы приступить к написанию RNN. Для реализации возьмем за основу модуль `nn.RNNCell` — ячейку рекуррентной нейронной сети, которая выполняет один шаг обработки последовательности.

$$
h_{t+1} = \tanh\left(W_{ih} x_t + b_{ih} + W_{hh} h_t + b_{hh}\right).
$$

In [ ]:
import torch
from torch import nn

B, D_input, D_hidden = 5, 10, 20

rnn_cell = nn.RNNCell(D_input, D_hidden)

x = torch.rand(B, D_input)
h = torch.rand(B, D_hidden)

h1 = rnn_cell(x, h)
h1.shape

Сравним с результатом, посчитанным вручную.

In [ ]:
h2 = torch.tanh(
    nn.functional.linear(x, weight=rnn_cell.weight_ih, bias=rnn_cell.bias_ih)
    + nn.functional.linear(h, weight=rnn_cell.weight_hh, bias=rnn_cell.bias_hh)
)
h2.shape

In [ ]:
torch.allclose(h1, h2)

Таким образом, на каждом шаге мы будем получать обновленное скрытое состояние модели. Нам останется лишь спроецировать его на словарь. Для этого нужен линейный слой с матрицей весов размера $|V| \times D_{\text{hidden}}$.

Заметим следующее: если размерность входных эмбеддингов $D_{\text{input}}$ равна размерности скрытых состояний модели $D_{\text{hidden}}$, матрица эмбеддингов также имеет размер $|V| \times D_{\text{hidden}}$. А значит, мы можем использовать ее веса для проекции скрытых состояний на словарь. Такой прием называется **связыванием весов** (англ. *weight tying*) и широко применяется в индустрии.

In [ ]:
import torch
from lightning import LightningModule
from lightning.pytorch.utilities.types import OptimizerLRScheduler
from torch import nn


class MicroRNN(LightningModule):
    """Рекуррентная нейронная сеть.

    Условные обозначения:
        B - размер батча;
        D - размер эмбеддинга `embedding_dim`;
        L - длина входной последовательности;
        V - размер словаря.
    """

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        bos_token_id: int = 0,
        eos_token_id: int = 1,
        pad_token_id: int = 2,
    ) -> None:
        super().__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim

        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_token_id)
        self.rnn = nn.RNNCell(embedding_dim, embedding_dim)

    def training_step(self, batch: torch.Tensor, batch_idx: int | None = None) -> torch.Tensor:
        # [BOS] char1 char2 ... charN
        inputs = batch[:, :-1]  # (B, L)

        # char1 char2 ... charN [EOS]
        labels = batch[:, 1:]  # (B, L)

        inputs_embeds = self.embedding(inputs)  # (B, L, D)
        hidden_states = []

        # инициализируем начальное скрытое состояние нулями
        h = inputs_embeds.new_zeros((inputs_embeds.shape[0], self.embedding_dim))  # (B, D)

        for t in range(inputs_embeds.shape[1]):
            x = inputs_embeds[:, t]
            h = self.rnn(x, h)

            hidden_states.append(h)

        hidden_states = torch.stack(hidden_states, dim=1)

        # проецируем скрытые состояния на словарь
        logits = nn.functional.linear(hidden_states, self.embedding.weight)  # (B, L, V)

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, self.vocab_size),
            labels.reshape(-1),
            ignore_index=self.pad_token_id,
        )
        self.log(
            "loss",
            loss,
            prog_bar=True,
            on_step=True,
            on_epoch=True,
            batch_size=(labels != self.pad_token_id).sum().item(),
        )

        return loss

    def configure_optimizers(self) -> OptimizerLRScheduler:
        return torch.optim.AdamW(self.parameters(), lr=2.5e-3)

In [ ]:
from lightning import Trainer

model = MicroRNN(vocab_size=len(VOCABULARY), embedding_dim=128)

trainer = Trainer(max_epochs=20, gradient_clip_val=1.0)
trainer.fit(model, train_dataloaders=dataloader)

Теперь, когда у нас есть обученная модель, давайте попробуем генерировать новые последовательности. Пусть нам дан набор логитов для текущего шага. С помощью softmax мы можем преобразовать их в распределение вероятностей. Затем необходимо каким-то образом выбрать следующий токен и добавить его к уже сгенерированной последовательности.

С одной стороны, мы можем на каждом шаге брать токен с максимальной вероятностью. Такой подход называется **жадной генерацией** (англ. *greedy decoding*). Но тогда при одинаковом начальном контексте мы всегда будем получать один и тот же результат.

Чтобы это исправить, можно добавить в генерацию немного стохастичности — случайно выбирать следующий токен из полученного распределения вероятностей (англ. *sampling*).

Более того, мы можем сглаживать это распределение или делать его более острым за счет **температуры** (англ. *temperature*). Для этого перед применением softmax разделим логиты на температуру $T > 0$. Чем ближе температура к нулю, тем острее становится распределение. Напротив, чем больше температура, тем ближе распределение к равномерному. Давайте рассмотрим пример.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axs = plt.subplots(1, 3, sharey=True)
fig.set_size_inches(12, 4)

logits = torch.tensor([-1, 1, 2, 3, -2])

for ax, temperature in zip(axs, (0.01, 1.0, 100)):
    probabilities = nn.functional.softmax(logits / temperature, dim=0)

    sns.barplot(probabilities, ax=ax)
    ax.set_title(f"T = {temperature}")

In [ ]:
@torch.inference_mode()
def generate(
    model: MicroRNN,
    max_new_tokens: int = 50,
    enable_sampling: bool = False,
    temperature: float = 1.0,
    device: str | torch.device = "cpu",
) -> list[int]:
    model = model.eval().to(device)

    generated = []

    token = torch.tensor([model.bos_token_id], device=device)

    h = torch.zeros((1, model.embedding_dim), device=device)

    for _ in range(max_new_tokens):
        x = model.embedding(token)
        h = model.rnn(x, h)

        logits = nn.functional.linear(h, model.embedding.weight)
        logits[:, [model.bos_token_id, model.pad_token_id]] = -torch.inf

        probabilities = torch.softmax(logits / temperature, dim=-1)

        if enable_sampling:
            token = torch.multinomial(probabilities, num_samples=1).squeeze(-1)
        else:
            token = logits.argmax(dim=-1)

        token_id = token.item()

        if token_id == model.eos_token_id:
            break

        generated.append(token_id)

    return generated

In [ ]:
for _ in range(10):
    generated = generate(model, enable_sampling=False)
    print(decode(generated))

In [ ]:
for _ in range(10):
    generated = generate(model, enable_sampling=True, temperature=0.1)
    print(decode(generated))

In [ ]:
for _ in range(10):
    generated = generate(model, enable_sampling=True, temperature=3.0)
    print(decode(generated))

In [ ]:
for _ in range(10):
    generated = generate(model, enable_sampling=True, temperature=1.0)
    print(decode(generated))